# Step 1 – Import Libraries  
We start by importing all the required Python libraries.  
- **NumPy/Pandas** for data manipulation  
- **Matplotlib** for visualization  
- **scikit-learn** for preprocessing, modeling, and evaluation  
- **XGBoost/LightGBM** if available (powerful gradient boosting models)  

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

import numpy as np
np.seterr(all="ignore")
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import make_scorer
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV,ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Optional gradient boosting libs
try:
    from xgboost import XGBRegressor
except:
    XGBRegressor = None

try:
    from lightgbm import LGBMRegressor
except:
    LGBMRegressor = None

print("All imports successful")

# Step 2 – List Dataset Files  
Check which files are available in the Kaggle competition dataset directory.  
This confirms that `train.csv`, `test.csv`, and other files are present.  

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Step 3 – Load Data  
Load the training and test sets using pandas.  
- `train.csv` contains features and the target (`SalePrice`)  
- `test.csv` contains features only (target hidden for scoring)  

In [ ]:
train_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_df  = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

# Step 4 – Explore Target Distribution  
We analyze the target variable `SalePrice`:  
- Plot raw distribution of sale prices.  
- Apply a log transform (`log1p`) to reduce skewness, matching Kaggle’s evaluation metric.  

In [ ]:
fig, ax = plt.subplots()
train_df['SalePrice'].hist(ax=ax, bins=50)
ax.set_title('SalePrice distribution (original)')
plt.show()

fig, ax = plt.subplots()
np.log1p(train_df['SalePrice']).hist(ax=ax, bins=50)
ax.set_title('SalePrice distribution (log1p)')
plt.show()

# Step 5 – Check Missing Values  
Visualize the top 20 features with missing data in the training set.  
This helps identify which columns need imputation or special handling.  

In [ ]:
missing = train_df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0][:20]

missing.plot(kind='bar', figsize=(10,4), title="Top 20 missing features")
plt.show()

# Step 6 – Feature Engineering  
We create new meaningful features to improve model performance:  
- Convert numeric codes (`MSSubClass`, `MoSold`, `YrSold`) to categorical strings.  
- Add composite features like `TotalSF` (total square footage), `Baths`, `PorchSF`.  
- Compute house age and remodel indicators.  
- Create `OverallScore` combining quality and condition.  

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # Treat codes as categories
    for col in ['MSSubClass','MoSold','YrSold']:
        if col in out.columns:
            out[col] = out[col].astype(str)
    # Composite areas
    out['TotalSF'] = out.get('TotalBsmtSF',0) + out.get('1stFlrSF',0) + out.get('2ndFlrSF',0)
    out['Baths'] = out.get('FullBath',0) + 0.5*out.get('HalfBath',0) + out.get('BsmtFullBath',0) + 0.5*out.get('BsmtHalfBath',0)
    out['PorchSF'] = out.get('OpenPorchSF',0) + out.get('EnclosedPorch',0) + out.get('3SsnPorch',0) + out.get('ScreenPorch',0)
    # Age features
    out['HouseAge'] = out['YrSold'].astype(int) - out.get('YearBuilt',0)
    out['SinceRemod'] = out['YrSold'].astype(int) - out.get('YearRemodAdd',0)
    out['IsRemodeled'] = (out.get('YearRemodAdd',0) != out.get('YearBuilt',0)).astype(int)
    # Quality composite
    if 'OverallQual' in out and 'OverallCond' in out:
        out['OverallScore'] = out['OverallQual'] + out['OverallCond']
    return out

train_fe = add_features(train_df)
test_fe = add_features(test_df)

from scipy.stats import skew

# Log-transform skewed numerical features
numeric_feats = train_fe[num_cols].dtypes[train_fe[num_cols].dtypes != "object"].index
skewed_feats = train_fe[numeric_feats].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
skewed_feats = skewed_feats[abs(skewed_feats) > 0.75].index

for feat in skewed_feats:
    train_fe[feat] = np.log1p(train_fe[feat])
    test_fe[feat] = np.log1p(test_fe[feat])


# Step 7 – Prepare Features and Target  
- Define target `y_log = log1p(SalePrice)`.  
- Split into numeric and categorical features.  
- Align test data columns with training features to avoid mismatches.  

In [ ]:
# Remove outliers
train_fe = train_fe.drop(train_fe[(train_fe['GrLivArea'] > 4000) & (train_fe['SalePrice'] < 300000)].index)
train_fe = train_fe.drop(train_fe[train_fe['LotArea'] > 100000].index)

y_log = np.log1p(train_fe['SalePrice'])
X = train_fe.drop(columns=['SalePrice'])

num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if X[c].dtype == 'object']

# Align test columns
for c in X.columns:
    if c not in test_fe.columns:
        test_fe[c] = np.nan
test_fe = test_fe[X.columns]

# Step 8 – Preprocessing Pipelines  
Use scikit-learn pipelines for consistent preprocessing:  
- **Numeric**: median imputation.  
- **Categorical**: impute most frequent value, then one-hot encode.  

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler())
])

categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocess = ColumnTransformer([
    ('num', numeric_pipe, num_cols),
    ('cat', categorical_pipe, cat_cols)
])

# Step 9 – Define Models  
We set up multiple regression algorithms to compare:  
- Linear Regression  
- Ridge (regularized linear)  
- Lasso (regularized linear)  
- Random Forest  
- Gradient Boosting  
- XGBoost (if available)  
- LightGBM (if available)  

In [ ]:
models = {
    'LassoCV': LassoCV(alphas=np.logspace(-4,0,50), max_iter=50000, cv=10, random_state=42),
    'RidgeCV': RidgeCV(alphas=np.logspace(-3,3,50), cv=10),
    'ElasticNet': ElasticNet(alpha=0.001, l1_ratio=0.7, max_iter=50000, random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=1200, max_depth=None, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(
        n_estimators=3000, learning_rate=0.01, max_depth=3, subsample=0.8, random_state=42
    )
}

if XGBRegressor:
    models['XGBoost'] = XGBRegressor(
        n_estimators=4000, learning_rate=0.01, max_depth=3,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, tree_method='hist'
    )
if LGBMRegressor:
    models['LightGBM'] = LGBMRegressor(
        n_estimators=6000, learning_rate=0.01, num_leaves=31,
        subsample=0.8, colsample_bytree=0.8,
        min_child_samples=20, feature_fraction=0.8,
        random_state=42, n_jobs=-1
    )


# Step 10 – Cross-Validation  
Evaluate each model using 5-fold cross-validation with **RMSE on log(SalePrice)**.  
- Collect mean and std RMSE for each model.  
- Lower RMSE indicates better performance.  

In [ ]:
warnings.filterwarnings("ignore")
np.seterr(all="ignore")

def rmse(y_true, y_pred): return np.sqrt(np.mean((y_true - y_pred)**2))
log_rmse_scorer = make_scorer(rmse, greater_is_better=False)

with np.errstate(all="ignore"):
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    results = []

    for name, model in models.items():
        pipe = Pipeline([('prep', preprocess), ('model', model)])
        scores = cross_val_score(pipe, X, y_log, scoring=log_rmse_scorer, cv=cv, n_jobs=-1)
        rmse_scores = -scores
        results.append({'Model': name, 'CV_RMSE_Mean': rmse_scores.mean(), 'CV_RMSE_Std': rmse_scores.std()})
        print(f"{name}: {rmse_scores.mean():.5f}")

results_df = pd.DataFrame(results).sort_values('CV_RMSE_Mean')
results_df

# Step 11 – Compare Models Visually  
Plot a horizontal bar chart of CV RMSE results to see which model performs best.  

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.barh(results_df['Model'], results_df['CV_RMSE_Mean'])
ax.set_xlabel('CV RMSE (log SalePrice)')
ax.set_title('Model comparison')
plt.show()

# Step 12 – Train Best Model  
Select the model with the lowest CV RMSE and fit it on the entire training data.  
This final trained model will be used for prediction on the test set.  

In [ ]:
best_name = results_df.iloc[0]['Model']
if 'Ensemble' in models:
    best_name = 'Ensemble'   # prefer ensemble if present

print("Best model:", best_name)
best_model = models[best_name]
best_pipe = Pipeline([('prep', preprocess), ('model', best_model)])
best_pipe.fit(X, y_log)

preds = []
weights = {
    'LassoCV': 0.25,
    'RidgeCV': 0.15,
    'ElasticNet': 0.25,
    'RandomForest': 0.2,
    'GradientBoosting': 0.15
}
print("Available models:", models.keys())


for name, weight in weights.items():
    model = models[name]
    pipe = Pipeline([('prep', preprocess), ('model', model)])
    pipe.fit(X, y_log)
    preds.append(weight * np.expm1(pipe.predict(test_fe)))

final_pred = np.sum(preds, axis=0)

submission = pd.DataFrame({'Id': test_df['Id'], 'SalePrice': final_pred})
submission.to_csv('/kaggle/working/submission.csv', index=False)


# Step 13 – Feature Importance  
Visualize the top 10 most important features of the selected model.  
This helps understand which variables drive house prices the most.  

In [ ]:
def plot_feature_importance(pipe, top_n=10):
    prep = pipe.named_steps['prep']
    model = pipe.named_steps['model']
    try:
        feat_names = []
        feat_names.extend(num_cols)
        ohe = prep.named_transformers_['cat'].named_steps['ohe']
        feat_names.extend(ohe.get_feature_names_out(cat_cols))
    except:
        return None
    
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
    elif hasattr(model, 'coef_'):
        importances = np.abs(model.coef_)
    else:
        return None
    
    fi = pd.DataFrame({'feature': feat_names, 'importance': importances})
    fi = fi.sort_values('importance', ascending=False).head(top_n)
    
    fi.plot(kind='barh', x='feature', y='importance', title=f'Top {top_n} Features ({best_name})', figsize=(8,4))
    plt.show()
    return fi

plot_feature_importance(best_pipe)

# Step 14 – Generate Submission File  
Predict house prices for the test set:  
- Invert the log transform with `expm1`.  
- Clip predictions to avoid negatives.  
- Save as `submission.csv` (format required by Kaggle).  
Finally, preview the submission file.  

In [ ]:
# Generate predictions
test_pred_log = best_pipe.predict(test_fe)
test_pred = np.expm1(test_pred_log)
test_pred = np.clip(test_pred, 1000, None)

# # Create submission DataFrame
# submission = pd.DataFrame({
#     'Id': test_df['Id'],
#     'SalePrice': test_pred
# })

# # Save locally
# submission.to_csv('submission2.csv', index=False)  # saves to your current folder

# # Preview
# print(submission.head())
